In [9]:
import torch
import pandas as pd
def cov(thetas, sigmas, N, ATT):

    ub = thetas + 1.96 * sigmas/ torch.sqrt(torch.tensor(N))
    lb = thetas - 1.96 * sigmas / torch.sqrt(torch.tensor(N))
    coverage = torch.mean( ( (ATT >= lb) & (ATT <= ub) ).float() , 0 )
    interval_length = torch.mean(ub - lb,0)
    return {"Coverage":coverage,
            "interval_length": interval_length}

In [10]:
N = 1500
# List of propensity models
models = ['logistic', 'truncated_logistic', 'truncated_adv']

# Create lists to store results
model_names = []
rmse_linear_list = []
sd_linear_list = []
rmse_riesz_list = []
sd_riesz_list = []
mea_linear_list = []
mea_riesz_list = []
coverage_linear_list = []
coverage_riesz_list = []
int_length_linear_list = []
int_length_riesz_list = []


# Read and compute RMSE, SD, Coverage, and Interval Length for each model
for model in models:
    file_suffix = f"N:{N}_{model}"

    pred_theta = torch.load(f"results/{file_suffix}_pred_theta.pt")
    pred_sig = torch.load(f"results/{file_suffix}_pred_sig.pt")
    ATT = torch.load(f"results/{file_suffix}_ATT_calculations.pt")["ATT"]


    rows = []

    for idx, method_name in zip([1, 3], ["Linear", "Dynamic Riesz"]):
        est = pred_theta[:, idx]
        sig = pred_sig[:, idx]

        bias = (est - ATT).mean().item()
        rmse = torch.sqrt(torch.mean((est - ATT) ** 2)).item()
        cov_result = cov(est, sig, N, ATT)
        coverage = cov_result["Coverage"].item()
        int_len = cov_result["interval_length"].item()

        rows.append([method_name, bias, rmse, coverage, int_len])

    df = pd.DataFrame(rows, columns=["Method", "Bias", "RMSE", "Coverage", "Interval Length"])
    
    print(model)
    print(df.to_string(index=False))



logistic
       Method      Bias     RMSE  Coverage  Interval Length
       Linear -0.055150 0.164860      0.76         0.395203
Dynamic Riesz -0.002838 0.156862      0.99         0.690919
truncated_logistic
       Method      Bias     RMSE  Coverage  Interval Length
       Linear -0.051190 0.165391      0.75         0.395243
Dynamic Riesz -0.004937 0.156204      0.99         0.690839
truncated_adv
       Method      Bias     RMSE  Coverage  Interval Length
       Linear -0.052733 0.166634      0.74         0.395211
Dynamic Riesz -0.002249 0.157142      0.99         0.691027
